# Paris Urban Mobility and Smart City Analytics Summary

## Step 1: Environment Setup and Data Ingestion
* Libraries Loaded: Initialized pandas, geopandas, numpy, matplotlib, contextily, and folium.
* Datasets Ingested: Loaded core Paris GeoJSON files into GeoDataFrames including quartier_paris, velib-emplacement-des-stations, fermetures-voirie, chantiers-perturbants, and arrondissements.
* Quality Assurance: Inspected data structures, verified CRS consistency, checked for missing or empty geometries, and validated primary keys.

## Step 2: Basic Spatial Operations and Filtering
* Attribute Filtering: Filtered neighborhoods by single and multiple arrondissement codes.
* CRS Transformations: Reprojected datasets to France Lambert-93 to establish metric copies for accurate spatial measurements.
* Geometry Metrics: Calculated surface areas, outer perimeters, and citywide land totals in meters.
* Base Map Management: Configured a master background map and rendered clean individual overlays using copy.deepcopy and custom CircleMarker point formatting.

## Step 3: Spatial Relationships and Spatial Joins
* Point-in-Polygon and Overlays: Performed spatial joins using predicates like within and intersects to map stations, construction sites, and road closures directly to their containing neighborhoods and arrondissements.
* Proximity and Buffers: Generated metric buffer zones around construction sites to isolate impacted bike docks.
* Nearest Neighbor Analysis: Computed exact distances from stations to nearby urban disruptions using sjoin_nearest.

## Step 4: Advanced Aggregations and Infrastructure Metrics
* Spatial Groupbys: Aggregated station counts, total dock capacities, and spatial densities across Paris neighborhoods.
* Infrastructure Audits: Identified mega-stations, under-sized stations prone to saturation, and calculated percentage-based land disruption per arrondissement caused by active construction.

## Step 5: Spatial Data Visualization and Map Creation
* Multi-Layer Mapping: Built advanced maps overlaying infrastructure features, construction polygons, and road closures together.
* Choropleth and Heatmaps: Created gradient-colored maps visualizing spatial variations in station density and dock capacity across administrative zones.
* Interactive Elements: Integrated hover tooltips, clickable station popups, and a UI layer toggle.
* Static and Export Formats: Generated publication-ready static plots with matplotlib and contextily basemaps, and exported interactive outputs to HTML.

# Step 1: Environment Setup & Data Ingestion

In [0]:
# installing libraries in Databricks
# %pip install geopandas
# %pip install geopandas matplotlib contextily
# %pip install folium

In [0]:
# import libraries
import pandas as pd
import geopandas as gpd
import numpy as np 
import matplotlib.pyplot as plt
import contextily as cx
import folium

In [0]:
# Load GeoJSON files into GeoDataFrames

gdf_quartiers = gpd.read_file('/Volumes/workspace/geojson_files/geojsons/quartier_paris.geojson')

gdf_velib = gpd.read_file('/Volumes/workspace/geojson_files/geojsons/velib-emplacement-des-stations.geojson')

gdf_fermeture = gpd.read_file('/Volumes/workspace/geojson_files/geojsons/fermetures-voirie.geojson')

gdf_chantiers = gpd.read_file('/Volumes/workspace/geojson_files/geojsons/chantiers-perturbants.geojson')

gdf_arrondissements = gpd.read_file('/Volumes/workspace/geojson_files/geojsons/arrondissements.geojson')




In [0]:
# inspect gdf_quartiers 
gdf_quartiers.head(5)

In [0]:
# inspect gdf velib 
gdf_velib.head(5)

In [0]:
# inspect gdf_arrondissements 
gdf_arrondissements.head(5)

In [0]:
# inspect gdf_chantiers
gdf_chantiers.head(5)

In [0]:
# inspect gdf_fermeture
gdf_fermeture.head(5)

In [0]:
# check the data type for arrondissemnts
gdf_arrondissements.dtypes

In [0]:
# check data type for velib 
gdf_velib.dtypes


In [0]:
# check the CRS of the geodataframes 
print(gdf_arrondissements.crs)
print(gdf_chantiers.crs)
print(gdf_fermeture.crs)
print(gdf_quartiers.crs)
print(gdf_velib.crs)

# all the GeoDataFrames have the same CRS

In [0]:
# geometry types in qurtiers
gdf_quartiers.geom_type.value_counts()


In [0]:
# geometry types  in velib
gdf_velib.geom_type.value_counts()

In [0]:
# Create a base Folium map centered on Paris
paris_map = folium.Map(
    location=[48.8566, 2.3522],
    zoom_start=12,
    tiles="OpenStreetMap"
)

# show the map 
paris_map

In [0]:
#  Count missing geometries (Null / None)
print(gdf_chantiers.geometry.isna().sum())

In [0]:
# Count empty geometries (e.g., POINT EMPTY)
print(gdf_chantiers.geometry.is_empty.sum())

In [0]:
#  Count duplicate station codes in Velib
gdf_velib["stationcode"].duplicated().sum()

# Step 2:  Spatial Operations & Filtering


In [0]:
# filter gdf_quartiers to select only the quartiers located in the 1st arrondissement
gdf_quartiers[gdf_quartiers['c_ar']==1]

In [0]:
# filter gdf_quartiers to select quartiers in MULTIPLE arrondissements at once (3,4,8)
gdf_quartiers[gdf_quartiers['c_ar'].isin([3,4,8])]

In [0]:
# transform the CRS of the geodataframes

gdf_quartiers_m = gdf_quartiers.to_crs(epsg=2154)
gdf_arrondissements_m = gdf_arrondissements.to_crs(epsg=2154)
gdf_chantiers_m = gdf_chantiers.to_crs(epsg=2154)
gdf_fermeture_m = gdf_fermeture.to_crs(epsg=2154)
gdf_velib_m = gdf_velib.to_crs(epsg=2154)
print(gdf_chantiers_m.crs)

In [0]:
# calculate the surface area of each quartier in square meters
gdf_quartiers_m["area_m2"] = gdf_quartiers_m.geometry.area
gdf_quartiers_m.head()

In [0]:
# calculate the outer perimeter of each quartier in meters
gdf_quartiers_m["perimeter_m"] = gdf_quartiers_m.geometry.length
gdf_quartiers_m.head(5)

In [0]:
# calculate the total land area of the entire city
gdf_quartiers_m["area_m2"].sum()

In [0]:
# Visualize quartiers in paris map 
import copy

map_quartiers = copy.deepcopy(paris_map)

folium.GeoJson(gdf_quartiers).add_to(map_quartiers)

# 3. Display the map
map_quartiers



In [0]:
# Visualize arrondissements in paris 

map_arrondissements = copy.deepcopy(paris_map)

folium.GeoJson(gdf_arrondissements).add_to(map_arrondissements)

# 3. Display the map
map_arrondissements

In [0]:
# Visualize velib stations in paris 

map_velib = copy.deepcopy(paris_map)

folium.GeoJson(
    gdf_velib,
    marker=folium.CircleMarker(radius=3, fill=True)
).add_to(map_velib)

# 3. Display the map
map_velib

In [0]:
# Visualize  chantiers in paris 

map_chantiers = copy.deepcopy(paris_map)

folium.GeoJson(
    gdf_chantiers.dropna(subset=["geometry"]).geometry
).add_to(map_chantiers)

# 3. Display the map
map_chantiers

# Step 3: Spatial Relationships & Spatial Joins

In [0]:
#  matches station points inside quartier polygons

velib_with_quartier = gpd.sjoin(gdf_velib, gdf_quartiers, how="inner", predicate="within")

display(velib_with_quartier[["name", "l_qu", "c_ar"]].head(5))

In [0]:
# keep ALL stations, even those outside Paris
all_velib_joined = gpd.sjoin(gdf_velib, gdf_quartiers, how="left", predicate="within")

# Filter where quartier name (l_qu) is missing (NaN)
velib_banlieue = all_velib_joined[all_velib_joined["l_qu"].isna()]
print("Stations in the suburbs:", len(velib_banlieue))

In [0]:
# matches station points inside arrondissement polygons

velib_with_arr = gpd.sjoin(gdf_velib, gdf_arrondissements, how="inner", predicate="within")

display(velib_with_arr[["stationcode", "name", "l_ar", "c_ar"]].head(5))

In [0]:
# count total stations per quartier

stations_per_quartier = velib_with_quartier.groupby("l_qu").size().reset_index(name="station_count")

display(stations_per_quartier.sort_values(by="station_count", ascending=False).head(10))

In [0]:
# sum total bike docks (capacity) per arrondissement

capacity_per_arr = velib_with_arr.groupby("l_ar")["capacity"].sum().reset_index()

display(capacity_per_arr.sort_values(by="capacity", ascending=False))

In [0]:
# match construction polygons with quartier polygons

chantiers_with_quartier = gpd.sjoin(gdf_chantiers, gdf_quartiers, how="inner", predicate="intersects")

display(chantiers_with_quartier[["l_qu", "c_ar"]].head(5))

In [0]:
# count total construction zones per quartier

chantiers_per_quartier = chantiers_with_quartier.groupby("l_qu").size().reset_index(name="chantiers_count")

display(chantiers_per_quartier.sort_values(by="chantiers_count", ascending=False).head(10))

In [0]:
# match road closures (gdf_fermeture) with arrondissements

fermeture_with_arr = gpd.sjoin(gdf_fermeture, gdf_arrondissements, how="inner", predicate="intersects")

# count road closures per arrondissement

fermeture_per_arr = fermeture_with_arr.groupby("l_ar").size().reset_index(name="fermeture_count")
display(fermeture_per_arr.sort_values(by="fermeture_count", ascending=False))

In [0]:
# find stations located directly inside a chantier polygon
stations_in_chantiers = gpd.sjoin(gdf_velib, gdf_chantiers, how="inner", predicate="within")

display(stations_in_chantiers[["stationcode", "name", "capacity"]])

In [0]:
# create a 50m buffer around metric construction zones (EPSG:2154)
chantiers_buffer = gdf_chantiers_m.dropna(subset=["geometry"]).copy()
chantiers_buffer["geometry"] = chantiers_buffer.geometry.buffer(50)

# find metric stations within 50m of construction
stations_near_chantiers = gpd.sjoin(gdf_velib_m, chantiers_buffer, how="inner", predicate="within")

# remove duplicate stations
stations_near_chantiers = stations_near_chantiers.drop_duplicates(subset=["stationcode"])
print("Stations within 50m of chantiers:", len(stations_near_chantiers))

In [0]:
# create a 30m buffer around metric road closures (EPSG:2154)
fermeture_buffer = gdf_fermeture_m.dropna(subset=["geometry"]).copy()
fermeture_buffer["geometry"] = fermeture_buffer.geometry.buffer(30)

# find metric stations within 30m of a closed road
stations_near_fermeture = gpd.sjoin(gdf_velib_m, fermeture_buffer, how="inner", predicate="within")
stations_near_fermeture = stations_near_fermeture.drop_duplicates(subset=["stationcode"])

display(stations_near_fermeture[["stationcode", "name", "capacity"]].head(5))

In [0]:
# find nearest chantier for every station and calculate distance in meters
nearest_chantier = gpd.sjoin_nearest(
    gdf_velib_m, 
    gdf_chantiers_m.dropna(subset=["geometry"]), 
    how="left", 
    distance_col="dist_meters"
)

display(nearest_chantier[["stationcode", "name", "dist_meters"]].sort_values(by="dist_meters").head(10))

In [0]:
# convert impacted stations back to EPSG:4326 for Folium
stations_impacted_4326 = stations_near_chantiers.to_crs(epsg=4326)

# Visualize impacted stations in paris map
map_impacted = copy.deepcopy(paris_map)

folium.GeoJson(
    stations_impacted_4326.geometry,
    marker=folium.CircleMarker(radius=4, color="red", fill=True)
).add_to(map_impacted)

# 3. Display the map
map_impacted

# Step 4:   Spatial Data Visualization & Map Creation



In [0]:
# Visualize road closures (gdf_fermeture) in red
map_fermeture = copy.deepcopy(paris_map)

folium.GeoJson(
    gdf_fermeture.dropna(subset=["geometry"]).geometry,
    style_function=lambda x: {"color": "red", "weight": 3}
).add_to(map_fermeture)

# Display the map
map_fermeture

In [0]:
# Choropleth map of quartier surface area (using area_m2 from Step 2)
map_choro_area = copy.deepcopy(paris_map)

folium.Choropleth(
    geo_data=gdf_quartiers.geometry.__geo_interface__,
    data=gdf_quartiers_m,
    columns=[gdf_quartiers_m.index, "area_m2"],
    key_on="feature.id",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name="Quartier Area (m2)"
).add_to(map_choro_area)

# 3. Display the map
map_choro_area

In [0]:
# Count stations per quartier and merge onto gdf_quartiers
counts = velib_with_quartier.groupby("l_qu").size().reset_index(name="station_count")
gdf_quartiers_count = gdf_quartiers.merge(counts, on="l_qu", how="left").fillna(0)

# Choropleth map of station count per quartier
map_choro_stations = copy.deepcopy(paris_map)

folium.Choropleth(
    geo_data=gdf_quartiers_count.geometry.__geo_interface__,
    data=gdf_quartiers_count,
    columns=[gdf_quartiers_count.index, "station_count"],
    key_on="feature.id",
    fill_color="YlOrRd",
    fill_opacity=0.7,
    line_opacity=0.3,
    legend_name="Number of Velib Stations"
).add_to(map_choro_stations)

# 3. Display the map
map_choro_stations

In [0]:
# Sum dock capacity per arrondissement and merge onto gdf_arrondissements
cap_arr = velib_with_quartier.groupby("c_ar")["capacity"].sum().reset_index()
gdf_arr_cap = gdf_arrondissements.merge(cap_arr, on="c_ar", how="left").fillna(0)

# Choropleth map of total bike dock capacity per arrondissement
map_choro_capacity = copy.deepcopy(paris_map)

folium.Choropleth(
    geo_data=gdf_arr_cap.geometry.__geo_interface__,
    data=gdf_arr_cap,
    columns=[gdf_arr_cap.index, "capacity"],
    key_on="feature.id",
    fill_color="BuPu",
    fill_opacity=0.7,
    line_opacity=0.4,
    legend_name="Total Dock Capacity per Arrondissement"
).add_to(map_choro_capacity)

# 3. Display the map
map_choro_capacity

In [0]:
# Interactive map showing arrondissement name and number on mouse hover
map_tooltip = copy.deepcopy(paris_map)

folium.GeoJson(
    gdf_arrondissements[["l_ar", "c_ar", "geometry"]],
    style_function=lambda x: {"fillColor": "#3186cc", "color": "black", "weight": 1.5, "fillOpacity": 0.3},
    tooltip=folium.GeoJsonTooltip(
        fields=["l_ar", "c_ar"],
        aliases=["Arrondissement:", "Code:"]
    )
).add_to(map_tooltip)

# 3. Display the map
map_tooltip

In [0]:
# Compare Large Stations (green) vs Small Stations (red)
map_station_sizes = copy.deepcopy(paris_map)

# Large stations (50+ docks)
folium.GeoJson(
    gdf_velib[gdf_velib["capacity"] >= 50].geometry,
    marker=folium.CircleMarker(radius=5, color="green", fill=True, fill_opacity=0.8)
).add_to(map_station_sizes)

# Small stations (<20 docks)
folium.GeoJson(
    gdf_velib[gdf_velib["capacity"] < 20].geometry,
    marker=folium.CircleMarker(radius=4, color="red", fill=True, fill_opacity=0.8)
).add_to(map_station_sizes)

# 3. Display the map
map_station_sizes

In [0]:
# Visualize Paris stations (blue) vs Suburb/Banlieue stations (orange)
map_paris_vs_banlieue = copy.deepcopy(paris_map)

# Inside Paris
folium.GeoJson(
    velib_with_quartier.geometry,
    marker=folium.CircleMarker(radius=2, color="blue", fill=True)
).add_to(map_paris_vs_banlieue)

# Outside Paris (Banlieue)
folium.GeoJson(
    velib_banlieue.geometry,
    marker=folium.CircleMarker(radius=3, color="orange", fill=True)
).add_to(map_paris_vs_banlieue)

# 3. Display the map
map_paris_vs_banlieue

In [0]:
# Overlay construction zones, road closures, and Velib stations on one map
map_disruption = copy.deepcopy(paris_map)

# 1. Construction zones in orange
folium.GeoJson(
    gdf_chantiers.dropna(subset=["geometry"]).geometry,
    style_function=lambda x: {"color": "darkorange", "fillColor": "orange", "fillOpacity": 0.5, "weight": 1}
).add_to(map_disruption)

# 2. Road closures in red
folium.GeoJson(
    gdf_fermeture.dropna(subset=["geometry"]).geometry,
    style_function=lambda x: {"color": "red", "weight": 3}
).add_to(map_disruption)

# 3. Velib stations in blue
folium.GeoJson(
    gdf_velib.geometry,
    marker=folium.CircleMarker(radius=2, color="blue", fill=True)
).add_to(map_disruption)

# 3. Display the map
map_disruption

In [0]:
# Interactive map with a LayerControl checkbox menu in the top right corner
map_dashboard = copy.deepcopy(paris_map)

folium.GeoJson(
    gdf_arrondissements.geometry, 
    name="Arrondissements",
    style_function=lambda x: {"fillOpacity": 0, "color": "black", "weight": 2}
).add_to(map_dashboard)

folium.GeoJson(
    gdf_quartiers.geometry, 
    name="Quartiers",
    style_function=lambda x: {"fillOpacity": 0, "color": "gray", "weight": 1}
).add_to(map_dashboard)

folium.GeoJson(
    gdf_chantiers.dropna(subset=["geometry"]).geometry, 
    name="Chantiers",
    style_function=lambda x: {"color": "orange"}
).add_to(map_dashboard)

folium.GeoJson(
    gdf_fermeture.dropna(subset=["geometry"]).geometry, 
    name="Fermetures",
    style_function=lambda x: {"color": "red", "weight": 3}
).add_to(map_dashboard)

folium.GeoJson(
    gdf_velib.geometry, 
    name="Velib Stations",
    marker=folium.CircleMarker(radius=2, color="blue", fill=True)
).add_to(map_dashboard)

# Add toggle control
folium.LayerControl().add_to(map_dashboard)

# 3. Display the map
map_dashboard